# 06. Optional late-vision LoRA

Run this last among training experiments. Vision LoRA adds backward computation through the
image encoder and has the highest memory risk on the 8 GiB GPU. Start with qkv/proj in only
the final four vision blocks; preserve the patch embedding, early blocks and original weights.
The helper supplies gradient flow into the upstream encoder's reentrant checkpoints.

Vision-only separates image adaptation from decoder adaptation. Decoder + vision checks their
combination. An all-three experiment is available after evidence supports it. No full-model
fine-tuning or automatic bitsandbytes/quantization change is performed.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
pages = H.read_pages()
train_pages, val_pages = H.partition(pages, "train"), H.partition(pages, "val")
RUN_VISION = False  # Set True only after 03–05 work and you have memory headroom.
EXPERIMENTS = [
    H.experiment("vision_last4_r4", decoder=None, vision=True, vision_last_layers=4),
    H.experiment("decoder_qv_last8_vision_last4_r4", decoder="qv", vision=True, vision_last_layers=4),
]
# Optional, after reviewing the independent comparisons:
# EXPERIMENTS.append(H.experiment("decoder_projector_vision_r4", decoder="qv",
#                               projector=True, vision=True, vision_last_layers=4))
if RUN_VISION:
    for config in EXPERIMENTS:
        directory = H.train_run(config, train_pages, val_pages)
        display(json.loads((directory / "summary.json").read_text()))
else:
    print("Vision training disabled. Review the earlier experiments before enabling it.")
